### Import Dependencies

In [1]:
import openai
import instructor
from pydantic import BaseModel, Field

from qdrant_client import QdrantClient

In [2]:
from dotenv import load_dotenv

load_dotenv("../../.env")

True

### Mock Example

In [3]:
prompt = """You are a helpful assistant.
Return an answer to the question
Question: what is your name?"""

In [6]:
response = openai.chat.completions.create(
    model="gpt-6-luna",
    messages=[
        {"role": "system", "content": prompt}
    ],
    reasoning_effort="none"
)

print(response.choices[0].message.content)

I’m ChatGPT.


In [5]:
response

ChatCompletion(id='chatcmpl-EelGMX6SHxkgV3QsSkjbeiVusH5Ir', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='I’m ChatGPT.', refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=None))], created=1793383506, model='gpt-5.4-nano-2026-03-17', object='chat.completion', metadata=None, moderation=None, service_tier='default', system_fingerprint=None, usage=CompletionUsage(completion_tokens=8, prompt_tokens=26, total_tokens=34, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=0, rejected_prediction_tokens=0, text_tokens=None), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cache_write_tokens=None, cached_tokens=0, image_tokens=None, text_tokens=None)))

### Add Instructor (Structured Outputs)

In [7]:
client = instructor.from_provider(
    "openai/gpt-6-luna",
    mode=instructor.Mode.RESPONSES_TOOLS
)

In [8]:
class Answer(BaseModel):
    answer: str = Field(description="Answer to the question")

In [9]:
response =  client.create(
    messages=[
        {"role": "system", "content": prompt}
    ],
    reasoning={"effort": "none"},
    response_model=Answer
)

In [14]:
response

Answer(answer='My name is ChatGPT.')

In [10]:
response, raw_response =  client.create_with_completion(
    messages=[
        {"role": "system", "content": prompt}
    ],
    reasoning={"effort": "none"},
    response_model=Answer
)

In [11]:
response

Answer(answer='My name is ChatGPT.')

In [12]:
raw_response

Response(id='resp_0bb4a2a283778ae5006ae4dc765b8087d2933e9981c5c7ac0a', created_at=1793383542.0, error=None, incomplete_details=None, instructions=None, metadata={}, model='gpt-6-luna', object='response', output=[ResponseFunctionToolCall(arguments='{"answer":"My name is ChatGPT."}', call_id='call_anTrBUqhOROyDHU45X9ldG5D', name='Answer', type='function_call', id='fc_0bb4a2a283778ae5006ae4dc77125c87d299c1ca55bf0e7f82', caller=None, namespace=None, status='completed')], parallel_tool_calls=True, temperature=1.0, tool_choice=ToolChoiceFunction(name='Answer', type='function'), tools=[FunctionTool(name='Answer', parameters={'properties': {'answer': {'description': 'Answer to the question', 'title': 'Answer', 'type': 'string'}}, 'required': ['answer'], 'title': 'Answer', 'type': 'object', 'additionalProperties': False}, strict=True, type='function', allowed_callers=None, defer_loading=None, description='Correctly extracted `Answer` with all the required parameters with correct types', output_

In [13]:
raw_response.usage

ResponseUsage(input_tokens=118, input_tokens_details=InputTokensDetails(cache_write_tokens=0, cached_tokens=0), output_tokens=23, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=141)

In [15]:
class AnswerWithReasoning(BaseModel):
    reasoning: str = Field(description="Reasoning for the answer")
    answer: str = Field(description="Answer to the question")

In [16]:
response, raw_response =  client.create_with_completion(
    messages=[
        {"role": "system", "content": prompt}
    ],
    reasoning={"effort": "none"},
    response_model=AnswerWithReasoning
)

In [17]:
response

AnswerWithReasoning(reasoning='The user asks for my name. I can identify myself simply as ChatGPT.', answer='My name is ChatGPT.')

### RAG Pipeline

In [18]:
class RAGGenerationResponse(BaseModel):
    answer: str = Field(description="Answer to the question")

In [21]:
qdrant_client = QdrantClient("url=http://localhost:6333")

def get_embedding(text, model="text-embedding-3-small"):
    response = openai.embeddings.create(
        input=text,
        model=model
    )

    return response.data[0].embedding


def retrieve_data(query, qdrant_client, k=5):

    query_embedding = get_embedding(query)

    results = qdrant_client.query_points(
        collection_name="toys_and_games_small_sample",
        query=query_embedding,
        limit=k
    )

    retrieved_context_ids = []
    retrieved_context = []
    similarity_scores = []
    retrieved_context_ratings = []

    for result in results.points:
        retrieved_context_ids.append(result.payload["parent_asin"])
        retrieved_context.append(result.payload["all_description"])
        similarity_scores.append(result.score)
        retrieved_context_ratings.append(result.payload["average_rating"])

    return {
        "retrieved_context_ids": retrieved_context_ids,
        "retrieved_context": retrieved_context,
        "similarity_scores": similarity_scores,
        "retrieved_context_ratings": retrieved_context_ratings
    }


def process_context(context):

    formatted_context = ""

    for id, chunk, rating in zip(context["retrieved_context_ids"], context["retrieved_context"], context["retrieved_context_ratings"]):
        formatted_context += f"- ID: {id}, rating: {rating}, description: {chunk}\n"

    return formatted_context


def build_prompt(preprocessed_context, query):

    prompt = f"""
You are a shopping assistant that can answer questions about the products in stock.

You will be given a question and a list of context.

Instructions:
- Answer the question based on the provided context only.
- Never use word context and refer to it as the available products.
- Do not use markdown formatting.

Context:
{preprocessed_context}

Question:
{query}    
"""

    return prompt


def generate_answer(prompt):

    response, raw_response =  client.create_with_completion(
        messages=[
            {"role": "system", "content": prompt}
        ],
        reasoning={"effort": "none"},
        response_model=RAGGenerationResponse
    )

    return response


def rag_pipeline(query, qdrant_client, top_k=5):

    retrieved_context = retrieve_data(query, qdrant_client, top_k)
    preprocessed_context = process_context(retrieved_context)
    prompt = build_prompt(preprocessed_context, query)
    answer = generate_answer(prompt)

    final_answer = {
        "data_object": answer,
        "answer": answer.answer,
        "question": query,
        "retrieved_context_ids": retrieved_context["retrieved_context_ids"],
        "retrieved_context": retrieved_context["retrieved_context"]
    }

    return final_answer

In [24]:
output = rag_pipeline("Any USB chargeable toy for boy?", qdrant_client)

In [25]:
output

{'data_object': RAGGenerationResponse(answer='Yes. The Kidstinct RC Tank is USB-chargeable and includes a USB charger and two rechargeable batteries. The LANSAND RC Excavator also charges via USB and comes with two rechargeable batteries. Both are suitable toy options for a boy.'),
 'answer': 'Yes. The Kidstinct RC Tank is USB-chargeable and includes a USB charger and two rechargeable batteries. The LANSAND RC Excavator also charges via USB and comes with two rechargeable batteries. Both are suitable toy options for a boy.',
 'question': 'Any USB chargeable toy for boy?',
 'retrieved_context_ids': ['B0BC8TJGQP',
  'B0BLHHD8MZ',
  'B09XWB3M5X',
  'B08GL635DH',
  'B0BZGXZBJK'],
 'retrieved_context': ["Kidstinct RC Tank for Kids 6 7 8-15 Year Old, 2 Head Infrared Stunt Car Shooting Water Bullets & Blowing Bubble, 360° Rotating, 2.4Ghz 4WD 1:16 Remote Control with Lights Music, (YKC-002-Blue) 🏎️2 HEADS RC TANK: 2-in-1 design, our rc cars set not only contains the bullet turret, but also th

In [26]:
print(output["answer"])

Yes. The Kidstinct RC Tank is USB-chargeable and includes a USB charger and two rechargeable batteries. The LANSAND RC Excavator also charges via USB and comes with two rechargeable batteries. Both are suitable toy options for a boy.
